In [1]:
import math

In [2]:
list_shapes = [
  [1, 256, 8],  # 0: EMB1
  [1, 32, 32],  # 1: EMB2
  [1, 16, 80],  # 2: EMB3
  [1, 55, 8],   # 3: EMEC1
  [1, 13, 32],  # 4: EMEC2
  [1, 8, 32],   # 5: EMEC3
  [1, 4, 8],    # 6: HEC1
  [1, 4, 8],    # 7: HEC2
  [1, 4, 8],    # 8: HEC3
  [1, 32, 8],   # 9: PSB
  [1, 32, 8],   # 10: PSE
  [1, 8, 8],    # 11: TileCal1
  [1, 8, 8],    # 12: TileCal2
  [1, 5, 8],    # 13: TileCal3
  [1, 4, 8],    # 14: TileExt1
  [1, 4, 8],    # 15: TileExt2
  [1, 2, 8]     # 16: TileExt3
]

In [3]:
# LF: more compact: you can collect layers with the same geometry in the same "block"
list_shapes = [
  [1, 256, 8],  # 0: EMB1
  [1, 32, 32],  # 1: EMB2
  [1, 16, 80],  # 2: EMB3
  [1, 55, 8],   # 3: EMEC1
  [1, 13, 32],  # 4: EMEC2
  [1, 8, 32],   # 5: EMEC3
  [3, 4, 8],    # 6: HEC1 - HEC2 - HEC3
  [2, 32, 8],   # 9: PSB - PSE
  [2, 8, 8],    # 11: TileCal1 - TileCal2
  [1, 5, 8],    # 13: TileCal3
  [2, 4, 8],    # 14: TileExt1 - TileExt2
  [1, 2, 8]     # 16: TileExt3
]

In [4]:
# for the config yalm in configs/
edge=0
bin_edges=[edge]
for l in list_shapes:
    edge += math.prod(l)
    bin_edges.append(edge)
print(bin_edges)

[0, 2048, 3072, 4352, 4792, 5208, 5464, 5560, 6072, 6200, 6240, 6304, 6320]


In [5]:
# for the config yalm in configs/models
list_edges = []

for l in list_shapes:
    edge = math.prod(l)
    list_edges.append(edge)
print(list_edges)

[2048, 1024, 1280, 440, 416, 256, 96, 512, 128, 40, 64, 16]


In [6]:
shape = sum(list_edges)
print(shape)

6320


In [7]:
list_patch_shape = [
  [1, 8, 1],    # EMB1   (8 * 1 = 8)
  [1, 4, 2],    # EMB2   (4 * 2 = 8)
  [1, 2, 4],    # EMB3   (2 * 4 = 8)
# LF: could consider adding an empty layer, then have a patch shape of 2
# LF: it won't make it more computationally expensive because the overall number of patches will be reduced
  [1, 1, 8],    # EMEC1  (1 * 8 = 8) - 1 because of 55
  [1, 1, 8],    # EMEC2  (1 * 8 = 8) - 1 because of 13 (prime number)
  [1, 1, 8],    # EMEC3  (1 * 8 = 8)
  [1, 2, 4],    # HEC1   (2 * 4 = 8)
  [1, 2, 4],    # HEC2   (2 * 4 = 8)
  [1, 2, 4],    # HEC3   (2 * 4 = 8)
  [1, 8, 1],    # PSB    (8 * 1 = 8)
  [1, 8, 1],    # PSE    (8 * 1 = 8)
  [1, 4, 2],    # TileCal1 (4 * 2 = 8)
  [1, 4, 2],    # TileCal2 (4 * 2 = 8)
  [1, 1, 8],    # TileCal3 (1 * 8 = 8) - 1 because of 5 (prime number)
  [1, 2, 4],    # TileExt1 (2 * 4 = 8)
  [1, 2, 4],    # TileExt2 (2 * 4 = 8)
  [1, 2, 4]     # TileExt3 (2 * 4 = 8)
]

In [8]:
# LF: more compact version: need to specify the patch shape for exh "block"
list_patch_shape = [
  [1, 8, 1],    # EMB1   (8 * 1 = 8)
  [1, 4, 2],    # EMB2   (4 * 2 = 8)
  [1, 2, 4],    # EMB3   (2 * 4 = 8)
  [1, 1, 8],    # EMEC1  (1 * 8 = 8) - 1 because of 55
  [1, 1, 8],    # EMEC2  (1 * 8 = 8) - 1 because of 13 (prime number)
  [1, 1, 8],    # EMEC3  (1 * 8 = 8)
  [1, 2, 4],    # HEC1-2-3   (2 * 4 = 8)
  [1, 8, 1],    # PSB-E    (8 * 1 = 8)
  [1, 4, 2],    # TileCal1-2 (4 * 2 = 8)
  [1, 1, 8],    # TileCal3 (1 * 8 = 8) - 1 because of 5 (prime number)
  [1, 2, 4],    # TileExt1-2 (2 * 4 = 8)
  [1, 2, 4]     # TileExt3 (2 * 4 = 8)
]

In [9]:
def verify_patch(list_shape, list_patch_shape):
    if len(list_shape) != len(list_patch_shape):
        print("Different shapes")
        return False
    
    previous_patch_dim = math.prod(list_patch_shape[0])
    
    for i, (shape, patch_shape) in enumerate(zip(list_shape, list_patch_shape)):
        for axis_idx, (s, p) in enumerate(zip(shape, patch_shape)):
            if s % p != 0:
                print(f"patch {p} of layer {i}, shape {s} not divisible")
                return False
        patch_dim = math.prod(patch_shape)
        if patch_dim != previous_patch_dim:
            print(f"product patch dim in layer {i} is different from previous")
            return False
        previous_patch_dim = patch_dim
    
    return patch_dim

In [10]:
verify_patch(list_shapes, list_patch_shape)

8

In [11]:
num_patches=[]
for i in range(len(list_shapes)):
    # LF: are your coordinates (eta, phi) or (x, y)?
    patch_eta = int(list_shapes[i][1]/list_patch_shape[i][1])
    patch_phi = int(list_shapes[i][2]/list_patch_shape[i][2])

    # LF: a "block" can have more than one layer
    patch_z = int(list_shapes[i][0]/list_patch_shape[i][0])
    num_patches.append([patch_z,patch_eta, patch_phi])
print(num_patches)

# LF: useful to know the total number of patches
print("Total number of patches is ", sum(math.prod(i) for i in num_patches))

[[1, 32, 8], [1, 8, 16], [1, 8, 20], [1, 55, 1], [1, 13, 4], [1, 8, 4], [3, 2, 2], [2, 4, 8], [2, 2, 4], [1, 5, 1], [2, 2, 2], [1, 1, 2]]
Total number of patches is  790


In [12]:
import math

def get_factors(n):
    """Returns all integer divisors of a number n."""
    factors = []
    for i in range(1, int(math.sqrt(n)) + 1):
        if n % i == 0:
            factors.append(i)
            if i != n // i:
                factors.append(n // i)
    return sorted(factors)

def find_valid_patches(list_shapes):
    print(f"Analyzing {len(list_shapes)} layers...\n")
    
    # Find all possible patch_dims for each layer
    layer_possible_patch_dims = []
    
    for i, shape in enumerate(list_shapes):
        eta, phi = shape[1], shape[2] # ignores index 0 (which is always '1')
        
        eta_factors = get_factors(eta)
        phi_factors = get_factors(phi)
        
        # All possible patch_dim combinations for this specific layer
        possible_patch_dims = set()
        for e in eta_factors:
            for p in phi_factors:
                possible_patch_dims.add(e * p)
                
        layer_possible_patch_dims.append(possible_patch_dims)
        
    # Find patch_dims that are valid for all layers simultaneously
    universal_patch_dims = layer_possible_patch_dims[0]
    for patch_dim in layer_possible_patch_dims[1:]:
        universal_patch_dims = universal_patch_dims.intersection(patch_dim)
        
    universal_patch_dims = sorted(list(universal_patch_dims))
    
    if not universal_patch_dims:
        print("No universal patch_dim found for these dimensions")
        print("need to apply Zero-Padding to some layers to make them compatible.")
        return
        
    print(f"Possible universal patch_dims found: {universal_patch_dims}\n")
    
#     # For each valid patch_dim, generate the suggested YAML configuration
#     for target_dim in universal_patch_dims:
#         print("="*50)
#         print(f" SUGGESTION FOR patch_dim: {target_dim}")
#         print("="*50)
#         print("list_patch_shape: [")
        
#         for i, shape in enumerate(list_shapes):
#             eta, phi = shape[1], shape[2]
#             eta_factors = get_factors(eta)
#             phi_factors = get_factors(phi)
            
#             best_patch = None
            
#             # Try to find the combination [1, E, P] that exactly matches the target_dim
#             for e in eta_factors:
#                 for p in phi_factors:
#                     if e * p == target_dim:
#                         # Here we just pick the first valid one found
#                         best_patch = [1, e, p]
#                         break
#                 if best_patch:
#                     break
                    
#             print(f"  {best_patch},  # Layer {i} (shape {shape})")
#         print("]\n")


find_valid_patches(list_shapes)

Analyzing 12 layers...

Possible universal patch_dims found: [1, 2, 4, 8]

